# Aula 03 — Regressão linear por OLS (Versão professor)

Notebook de condução docente para a terceira aula do treinamento inferencial.

## Finalidade desta versão

Esta edição foi pensada para o professor e, por isso, contém:

- explicações conceituais sobre ajuste por mínimos quadrados ordinários;
- notas de condução oral sobre interpretação de coeficientes;
- perguntas para leitura crítica do modelo;
- alertas contra uso mecânico do R²;
- conexão entre amostra saneada, ajuste e leitura dos resíduos.

## Resultado esperado da aula

Ao final da condução, a turma deve compreender que regressão não é uma caixa-preta: ela produz uma equação interpretável, com coeficientes, significância, graus de liberdade e erros que precisam ser lidos tecnicamente.

## Roteiro sugerido de tempo

- **0 a 6 min** — retomada da amostra saneada e motivo para modelagem;
- **6 a 15 min** — escolha da variável dependente e das explicativas;
- **15 a 28 min** — ajuste do modelo OLS;
- **28 a 38 min** — leitura da equação e dos coeficientes;
- **38 a 46 min** — leitura de R², R² ajustado, RMSE e teste F;
- **46 a 50 min** — introdução aos resíduos e ponte para a Aula 4.

## Estratégia didática

Nesta aula, o professor deve sustentar quatro mensagens centrais:

1. o modelo responde a uma pergunta econômica, não apenas estatística;
2. coeficiente tem sinal, magnitude e interpretação contextual;
3. R² sozinho não valida modelo;
4. resíduos são parte da leitura, não um detalhe descartável.

In [1]:
# Importações da aula.
# Professor: este é um bom ponto para explicar a continuidade do curso.
# A base é carregada, a coluna de valor unitário pode ser recomposta,
# e o núcleo desta aula passa a ser o serviço de regressão.

from __future__ import annotations

from pathlib import Path
from typing import Final

import pandas as pd

from servicos.carregamento import load_raw_dataset, resolve_project_root
from servicos.unitarizacao import add_unit_price_column
from servicos.regressao import (
    attach_predictions_and_residuals,
    build_coefficients_table,
    build_model_summary,
    fit_ols_regression,
)

## Nota de condução oral

Sugestão de fala:

“Se a Aula 1 colocou os dados em uma escala comparável e a Aula 2 melhorou a consistência da massa amostral, a Aula 3 tenta explicar o preço a partir das características do imóvel. Aqui começa propriamente a modelagem inferencial.”

In [2]:
# Mesma estratégia de localização da base das aulas anteriores.
# Isso preserva coerência didática e reduz ruído de infraestrutura.

DATASET_CANDIDATES: Final[tuple[str, ...]] = (
    'amostras_residencial35.csv',
    'amostrasresidencial35.csv',
    'amostras_residencial.csv',
)

TARGET_COLUMN: Final[str] = 'preco'
PREFERRED_FEATURES: Final[tuple[str, ...]] = (
    'areaprivativa',
    'vagas',
    'distanciacentrokm',
    'dist_praia',
)


def locate_default_dataset(project_root: Path) -> Path:
    """Localiza automaticamente a base padrão das aulas iniciais."""
    data_dir = project_root / 'data'

    for filename in DATASET_CANDIDATES:
        candidate = data_dir / filename
        if candidate.exists():
            return candidate

    searched = ', '.join(DATASET_CANDIDATES)
    raise FileNotFoundError(
        'Nenhum arquivo padrão foi encontrado na pasta `data`. '
        f'Arquivos procurados: {searched}.'
    )

## Etapa 1 — Resolver a base da aula

### Intenção docente

Antes de modelar, confirme a infraestrutura e a base de entrada.
Isso ajuda a turma a entender que modelagem confiável depende tanto de especificação estatística quanto de organização correta dos dados.

In [3]:
project_root = resolve_project_root()
dataset_path = locate_default_dataset(project_root)

project_root, dataset_path

(PosixPath('/Users/elydocarmobarros/Desktop/ESTUDOS TEC/JUPYTER PROJECTS/treinamento_inferencia'),
 PosixPath('/Users/elydocarmobarros/Desktop/ESTUDOS TEC/JUPYTER PROJECTS/treinamento_inferencia/data/amostras_residencial.csv'))

## Etapa 2 — Carregar a base e preparar contexto

### O que dizer

Nesta aula, o preço total será tratado como variável dependente.
As variáveis explicativas escolhidas representam atributos que, em tese, ajudam a explicar economicamente o preço observado.

In [4]:
df_raw = load_raw_dataset(dataset_path)
df_model = add_unit_price_column(df_raw)

print('Dimensão da base:', df_model.shape)
df_model.head()

Dimensão da base: (20, 8)


,id,preco,areaprivativa,vagas,idadeaparente,distanciacentrokm,fontelink,valor_unitario
0,AP-001,750000.0,85.0,2,5.0,1.2,https://portalimoveis.com.br/anuncio/001,8823.529412
1,AP-002,820000.0,92.5,2,8.0,1.8,https://portalimoveis.com.br/anuncio/002,8864.864865
2,AP-003,690000.0,78.0,1,12.0,2.5,https://portalimoveis.com.br/anuncio/003,8846.153846
3,AP-004,1200000.0,115.0,3,2.0,0.8,https://portalimoveis.com.br/anuncio/004,10434.782609
4,AP-005,580000.0,65.0,1,20.0,4.2,https://portalimoveis.com.br/anuncio/005,8923.076923


### Nota ao professor

Se a turma perguntar por que modelar `preco` e não `valor_unitario`, use a pergunta como gancho didático.
Ela é válida e ajuda a discutir diferentes especificações possíveis de modelo.
Nesta implementação, seguimos a lógica do script da Aula 3, que usa `preco` como alvo principal. [cite:66]

## Etapa 3 — Escolher as variáveis explicativas disponíveis

### Intenção pedagógica

Nem toda variável desejada estará sempre presente na base.
Por isso, a seleção abaixo filtra automaticamente as variáveis preferidas que realmente existem no dataset.

In [5]:
available_features = [column for column in PREFERRED_FEATURES if column in df_model.columns]

print('Variável dependente:', TARGET_COLUMN)
print('Variáveis explicativas disponíveis:', available_features)

Variável dependente: preco
Variáveis explicativas disponíveis: ['areaprivativa', 'vagas', 'distanciacentrokm']


### Pergunta para a turma

“Uma variável economicamente desejável, mas ausente na base, pode ser inventada na modelagem?”

## Etapa 4 — Ajustar o modelo OLS

### Mensagem-chave

O ajuste por mínimos quadrados ordinários procura os coeficientes que minimizam a soma dos quadrados dos resíduos.
Em linguagem de sala: estamos encontrando a equação linear que melhor aproxima os preços observados segundo as variáveis disponíveis.

In [6]:
model = fit_ols_regression(
    df=df_model,
    target_col=TARGET_COLUMN,
    feature_columns=available_features,
)

model

OLSRegressionArtifacts(coefficients=const               -549422.227260
areaprivativa         16399.584411
vagas                 17567.447399
distanciacentrokm    -40893.354601
Name: coeficiente, dtype: float64, fitted_values=0     8.306053e+05
1     9.290662e+05
2     6.450794e+05
3     1.356518e+06
4     3.623661e+05
5     1.146329e+06
6     6.942359e+05
7     5.262352e+05
8     1.450784e+06
9     1.015175e+06
10    4.607214e+05
11    8.716254e+05
12    1.080688e+06
13    5.756452e+05
14    9.618442e+05
15    8.142902e+05
16    6.204589e+05
17    1.237631e+06
18    4.401902e+05
19    9.905118e+05
Name: valor_ajustado, dtype: float64, residuals=0     -80605.316932
1    -109066.187252
2      44920.582324
3    -156517.638495
4     217633.882486
5    -196328.998767
6      15764.071662
7     113764.766089
8     949216.433071
9    -135174.566051
10    159278.618592
11    -81625.399244
12   -170688.418554
13   -255645.199995
14   -121844.234789
15    -44290.217662
16     69541.080225
17   -2

### Nota de condução oral

Evite dizer apenas “o Python rodou a regressão”.
Diga algo como:

“Agora nós estimamos uma equação de preço. O software faz a conta, mas a leitura econômica e técnica continua sendo nossa responsabilidade.”

## Etapa 5 — Construir o resumo do modelo

### Objetivo docente

Aqui a turma passa do objeto matemático para a leitura interpretativa.
O resumo deve ajudar a responder: quantos dados entraram, quantos parâmetros foram estimados, quanto o modelo explica e qual é o erro típico de ajuste.

In [7]:
summary = build_model_summary(model)
summary

{'n_obs': 20,
 'n_params': 4,
 'gl_modelo': 3,
 'gl_residuos': 16,
 'r2': 0.5806689204730774,
 'r2_ajustado': 0.5020443430617794,
 'rmse': 257264.4866754095,
 'sse': 1323700322087.239,
 'ssr': 1832994677912.761,
 'sst': 3156695000000.0,
 'f_statistic': 7.385335980065158,
 'f_p_value': 0.0025263778927814284}

## Etapa 6 — Tabela de coeficientes

### O que destacar

Cada coeficiente deve ser lido em três dimensões:

- **sinal**: positivo ou negativo;
- **magnitude**: quanto altera o preço, ceteris paribus;
- **significância**: quão consistente é sua contribuição estatística individual.

In [8]:
coeff_table = build_coefficients_table(model)
coeff_table

,variavel,coeficiente,erro_padrao,estatistica_t,p_valor,significativo_10pct
0,const,-549422.227260,849629.975338,-0.646661,0.517852,NAO
1,areaprivativa,16399.584411,11491.457537,1.427111,0.153548,NAO
2,vagas,17567.447399,260162.988375,0.067525,0.946164,NAO
3,distanciacentrokm,-40893.354601,97769.177268,-0.418264,0.675754,NAO


### Fala sugerida

“Um coeficiente positivo indica associação positiva, e um coeficiente negativo indica associação negativa, mantidas as demais variáveis constantes. Mas o sinal, sozinho, não basta: precisamos ver magnitude e p-valor.”

## Etapa 7 — Montar uma forma legível da equação estimada

### Intenção didática

Transformar a tabela de coeficientes em equação textual ajuda a turma a perceber que regressão gera uma função explícita de preço.

In [9]:
def resolve_equation_terms(coeff_table: pd.DataFrame) -> str:
    """Gera uma representação textual da equação estimada."""
    pieces: list[str] = []

    for _, row in coeff_table.iterrows():
        variable = str(row['variavel'])
        coefficient = float(row['coeficiente'])

        if variable == 'const':
            pieces.append(f'{coefficient:,.2f}')
            continue

        signal = '+' if coefficient >= 0 else '-'
        pieces.append(f'{signal} {abs(coefficient):,.2f}·{variable}')

    return 'Preço = ' + ' '.join(pieces)


equation = resolve_equation_terms(coeff_table)
print(equation)

Preço = -549,422.23 + 16,399.58·areaprivativa + 17,567.45·vagas - 40,893.35·distanciacentrokm


### Pergunta para a turma

“Se a área privativa aumenta uma unidade, o que acontece com o preço estimado, mantidas as demais variáveis constantes?”

## Etapa 8 — Ler métricas globais do modelo

### Ponto conceitual

O professor deve separar claramente as métricas:

- **R²**: proporção explicada pelo modelo;
- **R² ajustado**: versão penalizada pela quantidade de variáveis;
- **RMSE**: tamanho típico do erro;
- **Teste F**: significância global do modelo.

O erro didático mais comum aqui é transformar R² em sinônimo de qualidade total do modelo.

In [10]:
print(f"Observações usadas no ajuste: {summary['n_obs']}")
print(f"Parâmetros estimados: {summary['n_params']}")
print(f"Graus de liberdade do modelo: {summary['gl_modelo']}")
print(f"Graus de liberdade dos resíduos: {summary['gl_residuos']}")
print(f"R²: {summary['r2']:.4f}")
print(f"R² ajustado: {summary['r2_ajustado']:.4f}")
print(f"RMSE: {summary['rmse']:,.2f}")
print(f"Teste F: {summary['f_statistic']:.4f}")
print(f"p-valor do teste F: {summary['f_p_value']:.6f}")

Observações usadas no ajuste: 20
Parâmetros estimados: 4
Graus de liberdade do modelo: 3
Graus de liberdade dos resíduos: 16
R²: 0.5807
R² ajustado: 0.5020
RMSE: 257,264.49
Teste F: 7.3853
p-valor do teste F: 0.002526


### Nota ao professor

Use este momento para dizer explicitamente:

- R² alto não salva especificação ruim;
- R² baixo não inutiliza automaticamente um modelo aplicado;
- RMSE traz a escala do erro para uma unidade economicamente mais intuitiva;
- teste F ajuda a saber se o modelo, como conjunto, tem relevância estatística.

## Etapa 9 — Acrescentar valores ajustados e resíduos

### Finalidade pedagógica

Esta etapa prepara a Aula 4.
Ao anexar predições e resíduos, a turma começa a enxergar onde o modelo acerta melhor e onde erra mais.

In [11]:
enriched_df = attach_predictions_and_residuals(df_model, model)
enriched_df.head()

,id,preco,areaprivativa,vagas,idadeaparente,distanciacentrokm,fontelink,valor_unitario,valor_ajustado,residuo
0,AP-001,750000.0,85.0,2,5.0,1.2,https://portalimoveis.com.br/anuncio/001,8823.529412,8.306053e+05,-80605.316932
1,AP-002,820000.0,92.5,2,8.0,1.8,https://portalimoveis.com.br/anuncio/002,8864.864865,9.290662e+05,-109066.187252
2,AP-003,690000.0,78.0,1,12.0,2.5,https://portalimoveis.com.br/anuncio/003,8846.153846,6.450794e+05,44920.582324
3,AP-004,1200000.0,115.0,3,2.0,0.8,https://portalimoveis.com.br/anuncio/004,10434.782609,1.356518e+06,-156517.638495
4,AP-005,580000.0,65.0,1,20.0,4.2,https://portalimoveis.com.br/anuncio/005,8923.076923,3.623661e+05,217633.882486


In [12]:
preview_columns = [
    column
    for column in ('id', 'preco', 'valor_ajustado', 'residuo')
    if column in enriched_df.columns
]

enriched_df[preview_columns].head(10)

,id,preco,valor_ajustado,residuo
0,AP-001,750000.0,8.306053e+05,-80605.316932
1,AP-002,820000.0,9.290662e+05,-109066.187252
2,AP-003,690000.0,6.450794e+05,44920.582324
3,AP-004,1200000.0,1.356518e+06,-156517.638495
4,AP-005,580000.0,3.623661e+05,217633.882486
5,AP-006,950000.0,1.146329e+06,-196328.998767
6,AP-007,710000.0,6.942359e+05,15764.071662
7,AP-008,640000.0,5.262352e+05,113764.766089
8,AP-009,2400000.0,1.450784e+06,949216.433071
9,AP-010,880000.0,1.015175e+06,-135174.566051


### Fala sugerida

“O resíduo é a parte do preço que o modelo não explicou. Na próxima aula, nós vamos avaliar se esse comportamento residual respeita as condições esperadas para uma boa inferência.”

## Etapa 10 — Leitura crítica inicial do ajuste

### Perguntas orientadoras

- os sinais dos coeficientes fazem sentido econômico?
- há variáveis com baixa significância individual?
- o modelo parece globalmente significativo?
- o erro típico parece aceitável para o problema estudado?
- os resíduos sugerem necessidade de diagnóstico mais profundo?

## Erros conceituais comuns

- acreditar que regressão substitui raciocínio técnico;
- interpretar coeficiente sem considerar unidade e contexto;
- usar apenas R² para avaliar o modelo;
- esquecer graus de liberdade;
- ignorar resíduos porque a equação “parece boa”.

## Exercício supervisionado

Peça à turma que redija, com base nas saídas do notebook:

- uma frase explicando o papel da variável dependente;
- uma frase interpretando um coeficiente escolhido;
- uma frase comentando o significado do teste F;
- uma frase justificando por que a análise ainda não terminou na Aula 3.

In [13]:
# Espaço livre para exploração guiada.
# Professor: use esta célula para pedir que a turma filtre casos,
# compare resíduos ou discuta sinais inesperados.

enriched_df.sort_values(by='residuo', ascending=False).head(10)

,id,preco,areaprivativa,vagas,idadeaparente,distanciacentrokm,fontelink,valor_unitario,valor_ajustado,residuo
8,AP-009,2400000.0,120.0,3,1.0,0.5,https://portalimoveis.com.br/anuncio/009,20000.000000,1.450784e+06,949216.433071
4,AP-005,580000.0,65.0,1,20.0,4.2,https://portalimoveis.com.br/anuncio/005,8923.076923,3.623661e+05,217633.882486
18,AP-019,610000.0,68.0,1,16.0,3.5,https://portalimoveis.com.br/anuncio/019,8970.588235,4.401902e+05,169809.781033
10,AP-011,620000.0,70.0,1,18.0,3.8,https://portalimoveis.com.br/anuncio/011,8857.142857,4.607214e+05,159278.618592
7,AP-008,640000.0,72.0,1,15.0,3.0,https://portalimoveis.com.br/anuncio/008,8888.888889,5.262352e+05,113764.766089
16,AP-017,690000.0,76.0,1,11.0,2.3,https://portalimoveis.com.br/anuncio/017,9078.947368,6.204589e+05,69541.080225
2,AP-003,690000.0,78.0,1,12.0,2.5,https://portalimoveis.com.br/anuncio/003,8846.153846,6.450794e+05,44920.582324
6,AP-007,710000.0,80.0,1,10.0,2.1,https://portalimoveis.com.br/anuncio/007,8875.000000,6.942359e+05,15764.071662
15,AP-016,770000.0,86.0,2,9.0,2.0,https://portalimoveis.com.br/anuncio/016,8953.488372,8.142902e+05,-44290.217662
0,AP-001,750000.0,85.0,2,5.0,1.2,https://portalimoveis.com.br/anuncio/001,8823.529412,8.306053e+05,-80605.316932


## Ponte para a Aula 4

Encerre com esta transição:

“Hoje nós ajustamos a equação e começamos a ler seus resultados. Na próxima aula, vamos verificar se os resíduos e os testes estatísticos sustentam tecnicamente o uso desse modelo.”